In [ ]:
!pip install pandas numpy rasterio scikit-learn lightgbm tqdm

In [ ]:
def load_gbif_txt(path):
    df = pd.read_csv(path, sep='\t', low_memory=False)

    # Handle different naming cases
    if 'species' not in df.columns:
        df['species'] = df['scientificName']

    df = df[['species', 'decimalLatitude', 'decimalLongitude']]
    df = df.dropna()

    df.columns = ['species', 'lat', 'lon']

    return df

In [ ]:
def clean_data(df):
    # Remove invalid coordinates
    df = df[(df['lat'] != 0) & (df['lon'] != 0)]

    # Keep only Lebanon region (IMPORTANT)
    df = df[
        (df['lat'] >= 33) & (df['lat'] <= 35) &
        (df['lon'] >= 35) & (df['lon'] <= 37)
    ]

    # Remove duplicates
    df = df.drop_duplicates()

    return df

In [ ]:
import rasterio
import numpy as np
import os

class ClimateExtractor:
    def __init__(self, raster_folder):
        self.folder = raster_folder
        self.rasters = []

        for i in range(1, 20):
            path = os.path.join(self.folder, f"wc2.1_10m_bio_{i}.tif")

            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing file: {path}")

            self.rasters.append(rasterio.open(path))

    def extract_point(self, lat, lon):
        values = []

        for raster in self.rasters:
            try:
                row, col = raster.index(lon, lat)
                val = raster.read(1)[row, col]
            except:
                val = np.nan

            values.append(val)

        return values

    # ✅ THIS IS WHAT YOU'RE MISSING
    def extract_dataframe(self, df):
      features = []

      total = len(df)
      for i, row in enumerate(df.itertuples(index=False)):
          vals = self.extract_point(row.lat, row.lon)
          features.append(vals)

          # Log every 1000 points
          if i % 1000 == 0:
              logging.info(f"Processed {i}/{total} points")

      feature_names = [f"BIO{i}" for i in range(1, 20)]
      return np.array(features), feature_names

In [ ]:
import numpy as np
import pandas as pd

def generate_pseudo_absence(df, n_samples):
    lat_min, lat_max = df['lat'].min(), df['lat'].max()
    lon_min, lon_max = df['lon'].min(), df['lon'].max()

    pseudo = pd.DataFrame({
        'species': df['species'].sample(n_samples, replace=True).values,
        'lat': np.random.uniform(lat_min, lat_max, n_samples),
        'lon': np.random.uniform(lon_min, lon_max, n_samples),
        'label': 0
    })

    return pseudo


def build_dataset(df, extractor):
    # Presence
    X_presence, feature_names = extractor.extract_dataframe(df)
    df_presence = pd.DataFrame(X_presence, columns=feature_names)
    df_presence['species'] = df['species'].values
    df_presence['label'] = 1

    # Pseudo absence
    pseudo_df = generate_pseudo_absence(df, len(df))
    X_absence, _ = extractor.extract_dataframe(pseudo_df)

    df_absence = pd.DataFrame(X_absence, columns=feature_names)
    df_absence['species'] = pseudo_df['species']
    df_absence['label'] = 0

    # Combine
    full_df = pd.concat([df_presence, df_absence], ignore_index=True)

    return full_df

In [ ]:
import lightgbm as lgb

def get_model():
    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        class_weight='balanced'
    )
    return model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import logging
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)

logging.info("Logging is working ✅")


logging.info("Loading GBIF dataset...")
gbif = load_gbif_txt("/content/drive/MyDrive/PhytoLB/dwca-tree_lebanon-v1.0/occurrence.txt")

logging.info(f"GBIF loaded: {len(gbif)} records")
logging.info(f"Unique species: {gbif['species'].nunique()}")


logging.info("Initializing Climate Extractor...")
extractor = ClimateExtractor("/content/drive/MyDrive/PhytoLB/wc2.1_10m_bio")

logging.info("Building dataset (this may take time)...")
dataset = build_dataset(gbif, extractor)

logging.info(f"Dataset built: {dataset.shape}")
logging.info(f"Class distribution:\n{dataset['label'].value_counts()}")

# Check missing values
missing = dataset.isna().sum().sum()
logging.info(f"Total missing values in dataset: {missing}")


logging.info("Splitting dataset...")
X = dataset.drop(columns=['label'])
y = dataset['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logging.info(f"Train size: {X_train.shape}")
logging.info(f"Test size: {X_test.shape}")


logging.info("Encoding species...")
le = LabelEncoder()

X_train['species_id'] = le.fit_transform(X_train['species'])
X_test['species_id'] = le.transform(X_test['species'])

X_train = X_train.drop(columns=['species'])
X_test = X_test.drop(columns=['species'])

logging.info(f"Encoded {len(le.classes_)} species")


logging.info("Imputing missing values...")
imputer = SimpleImputer(strategy='mean')

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Check if NaNs remain
logging.info(f"NaNs after imputation (train): {np.isnan(X_train).sum()}")
logging.info(f"NaNs after imputation (test): {np.isnan(X_test).sum()}")


logging.info("Training model...")
model = get_model()
model.fit(X_train, y_train)

logging.info("Model training completed")


logging.info("Evaluating model...")
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

logging.info(f"Accuracy: {acc:.4f}")
logging.info(f"AUC: {auc:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

2026-05-10 04:48:37,453 - INFO - Logging is working ✅
2026-05-10 04:48:37,464 - INFO - Loading GBIF dataset...
2026-05-10 04:48:37,753 - INFO - GBIF loaded: 13748 records
2026-05-10 04:48:37,759 - INFO - Unique species: 27
2026-05-10 04:48:37,762 - INFO - Initializing Climate Extractor...
2026-05-10 04:48:37,913 - INFO - Building dataset (this may take time)...
2026-05-10 04:48:40,598 - INFO - Processed 0/13748 points
2026-05-10 04:49:23,745 - INFO - Processed 1000/13748 points
2026-05-10 04:49:54,511 - INFO - Processed 2000/13748 points
2026-05-10 04:50:24,653 - INFO - Processed 3000/13748 points
2026-05-10 04:50:55,368 - INFO - Processed 4000/13748 points
2026-05-10 04:51:33,781 - INFO - Processed 5000/13748 points
2026-05-10 04:52:03,803 - INFO - Processed 6000/13748 points
2026-05-10 04:52:34,275 - INFO - Processed 7000/13748 points
2026-05-10 04:53:13,660 - INFO - Processed 8000/13748 points
2026-05-10 04:53:44,058 - INFO - Processed 9000/13748 points
2026-05-10 04:54:14,507 - INF

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


2026-05-10 05:03:06,482 - INFO - Accuracy: 0.9078
2026-05-10 05:03:06,483 - INFO - AUC: 0.9545

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.86      0.90      2750
           1       0.87      0.96      0.91      2750

    accuracy                           0.91      5500
   macro avg       0.91      0.91      0.91      5500
weighted avg       0.91      0.91      0.91      5500



In [ ]:
import os
import joblib

SAVE_DIR = "/content/drive/MyDrive/PhytoLB/model_enhanced_v1"

os.makedirs(SAVE_DIR, exist_ok=True)

logging.info("Saving extracted datasets...")

# Save full dataset
dataset.to_csv(f"{SAVE_DIR}/final_phyto_dataset.csv", index=False)

logging.info("Datasets saved successfully ✅")

2026-05-10 05:04:12,361 - INFO - Saving extracted datasets...
2026-05-10 05:04:12,829 - INFO - Datasets saved successfully ✅


In [ ]:
logging.info("Saving model artifacts...")

joblib.dump(model, f"{SAVE_DIR}/phyto_model.pkl")

joblib.dump(le, f"{SAVE_DIR}/species_encoder.pkl")

joblib.dump(imputer, f"{SAVE_DIR}/imputer.pkl")

logging.info("Model artifacts saved ✅")

2026-05-10 05:04:17,054 - INFO - Saving model artifacts...
2026-05-10 05:04:17,098 - INFO - Model artifacts saved ✅


In [ ]:
import pandas as pd

def predict_city(model, scaler, extractor, city_lat, city_lon, species_id):
    climate = extractor.extract_point(city_lat, city_lon)

    df = pd.DataFrame([climate], columns=[f"BIO{i}" for i in range(1, 20)])
    df['species_id'] = species_id

    X = scaler.transform(df)

    prob = model.predict_proba(X)[0][1]

    return prob